# Course 3 — Logistic Regression II

Multinomial logistic regression, L1/L2 regularization, and the moment
when a linear classifier runs out of road.

**Sections**
1. Multinomial logistic regression (0:00–0:30)
2. L2 and L1 regularization (0:30–1:00)
3. When linear fails — motivating non-linear classifiers (1:00–1:30)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_moons
iris = load_dataset('iris')
print(iris['species'].value_counts())


## 1. Multinomial logistic regression

Softmax generalizes the sigmoid: $P(y=k|x) = \frac{e^{\beta_k^\top x}}{\sum_j e^{\beta_j^\top x}}$.

In [ ]:
le = LabelEncoder().fit(iris['species'])
y = le.transform(iris['species'])
X = iris[['petal_length', 'petal_width']].to_numpy()
m = LogisticRegression(max_iter=2000).fit(X, y)
print('coef shape:', m.coef_.shape)  # (3, 2) — one row per class
print(f'training accuracy = {m.score(X, y):.4f}')


In [ ]:
x_min, x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
y_min, y_max = X[:,1].min()-0.5, X[:,1].max()+0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                      np.linspace(y_min, y_max, 200))
Z = m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
fig, ax = plt.subplots(figsize=(6, 5))
ax.contourf(xx, yy, Z, alpha=0.25, levels=[-0.5,0.5,1.5,2.5])
for cls in range(3):
    ax.scatter(X[y==cls,0], X[y==cls,1], label=le.classes_[cls], s=20)
ax.set_xlabel('petal_length'); ax.set_ylabel('petal_width'); ax.legend()
ax.set_title('Multinomial LR on iris'); plt.show()


## 2. L2 and L1 regularization

$C = 1/\lambda$. Small $C$ = strong regularization (coefficients shrunk).
Use `solver='liblinear'` or `'saga'` to enable L1.

In [ ]:
# Use full 4D iris this time; standardize for regularization to be fair.
X4 = iris[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].to_numpy()
Cs = np.logspace(-3, 2, 30)
paths_l2 = np.empty((len(Cs), 3, 4))
paths_l1 = np.empty((len(Cs), 3, 4))
scaler = StandardScaler().fit(X4)
Xs = scaler.transform(X4)
for i, C in enumerate(Cs):
    paths_l2[i] = LogisticRegression(C=C, max_iter=3000).fit(Xs, y).coef_
    paths_l1[i] = LogisticRegression(C=C, penalty='l1', solver='saga', max_iter=3000).fit(Xs, y).coef_

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
feat = ['sepal_len', 'sepal_wid', 'petal_len', 'petal_wid']
for ax, paths, title in zip(axes, [paths_l2, paths_l1], ['L2', 'L1']):
    # show class 0 (setosa) only to keep the plot readable
    for j in range(4):
        ax.plot(Cs, paths[:, 0, j], label=feat[j])
    ax.set_xscale('log'); ax.set_xlabel('C (= 1/λ)')
    ax.set_title(f'{title} coefficient paths (class: setosa)')
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


Notice that under L1 (right) some coefficients hit zero exactly for
small C — same sparsity story as Lasso in Week 3.

## 3. When linear fails

In [ ]:
Xm, ym = make_moons(n_samples=400, noise=0.25, random_state=0)
m = LogisticRegression(max_iter=2000).fit(Xm, ym)
print(f'LR on moons: accuracy = {m.score(Xm, ym):.4f}')


In [ ]:
xx, yy = np.meshgrid(np.linspace(Xm[:,0].min()-0.5, Xm[:,0].max()+0.5, 200),
                      np.linspace(Xm[:,1].min()-0.5, Xm[:,1].max()+0.5, 200))
Z = m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
fig, ax = plt.subplots(figsize=(6, 5))
ax.contourf(xx, yy, Z, alpha=0.25)
ax.scatter(Xm[ym==0,0], Xm[ym==0,1], s=10)
ax.scatter(Xm[ym==1,0], Xm[ym==1,1], s=10, color='C3')
ax.set_title('A line cannot separate two moons'); plt.show()


A *linear* boundary can do no better than ~85% here. We need either
non-linear features (next-week: SVM with an RBF kernel) or a non-linear
model (next-week: trees and ensembles).

### Recap
- Multinomial LR is the softmax classifier: one weight vector per class.
- C = 1/λ controls regularization; L1 induces sparsity, L2 just shrinks.
- Linear classifiers fundamentally cannot fit curved boundaries — that's
  the wall that motivates Day 2.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

**Task 1.** Fit plain multinomial logistic regression on `penguins`
to predict `species` from all four numeric measurements. Report 5-fold
CV accuracy. Drop NaN first.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
# your code here


### Exercise 1 — Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
df = load_dataset('penguins').dropna()
X = df[['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']]
y = df['species']
pipe = Pipeline([('s', StandardScaler()),
                 ('lr', LogisticRegression(max_iter=2000))])
print(f'CV accuracy = {cross_val_score(pipe, X, y, cv=5).mean():.4f}')


---

## Exercise 2

**Task 2.** Fit L1-penalized logistic regression with `C = 0.05`.
How many coefficients survive (are non-zero) for each class?

In [ ]:
# your code here


### Exercise 2 — Solution

In [ ]:
import numpy as np
Xs = StandardScaler().fit_transform(X)
m = LogisticRegression(C=0.05, penalty='l1', solver='saga',
                        max_iter=5000).fit(Xs, y)
for cls, row in zip(m.classes_, m.coef_):
    keep = [n for n, c in zip(X.columns, row) if c != 0]
    print(f'{cls:10s}: survivors = {keep}')


---

## Exercise 3

**Task 3.** Compare validation accuracy for `C ∈ [0.001, 0.01, 0.1, 1, 10]`
under L1. Plot accuracy vs C on a log axis.

In [ ]:
# your code here


### Exercise 3 — Solution

In [ ]:
import matplotlib.pyplot as plt
Cs = [0.001, 0.01, 0.1, 1, 10]
accs = []
for C in Cs:
    pipe = Pipeline([('s', StandardScaler()),
                     ('lr', LogisticRegression(C=C, penalty='l1', solver='saga',
                                               max_iter=5000))])
    accs.append(cross_val_score(pipe, X, y, cv=5).mean())
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogx(Cs, accs, marker='o')
ax.set_xlabel('C'); ax.set_ylabel('5-fold CV accuracy'); plt.show()
for C, a in zip(Cs, accs): print(f'C={C:7g}  acc = {a:.4f}')
